<h2><center> 🛡️ 👾 Cybersecurity Attacks & Defense – Analytical Threat Intelligence System in a Modern Cyber Battlefield 🐛 🌐 </center></h2>

<h4><center> From Raw Exchange To Security Insights: Machine - Deep Learning & Predictive Analytics & Real-World Protection </center></h4>

<p><b>Dataset Author:</b> Uneeb, Z. (2026). </p>

<p><b>Official Sources:</b>
<a href="https://otx.alienvault.com/" target="_blank">AlienVault OTX</a>,
<a href="https://www.cisa.gov/known-exploited-vulnerabilities-catalog" target="_blank">CISA KEV Catalog</a>,
<a href="https://nvd.nist.gov/" target="_blank">NVD (National Vulnerability Database)</a></p>

<p><b> Launch Date:</b> May 13rd - 14th, 2026 </p>

<p><b>Successors</b>:
    <ul>
        <li> Tolstoy, J. (2026). <i> CVE Ransomware Risk Modeling </i>. Kaggle. <a href="https://www.kaggle.com/code/tolstoyjustin/cve-ransomware-risk-modeling" target="_blank">[Repository]</a></li>
        <li> Tolstoy, J. (2026). <i> OTX NLP / MITRE ATT&CK Mapping </i>. Kaggle. <a href = "https://www.kaggle.com/code/tolstoyjustin/otx-nlp-mitre-att-ck-mapping"> [Repository]</a></li>
        <li> Uneeb, Z. (2026). <i> Cybersecurity Threat Intelligence - EDA & Analysis </i>. Kaggle. <a href = "https://www.kaggle.com/code/chuneeb/cybersecurity-threat-intelligence-eda-analysis">[Repository]</a></li>
    </ul>
</p>

<p><b>Hackathon Contestant:</b> Cresht </p>

### Import Libraries ###

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import sys, re, math
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.utils import normalize_text, safe_ratio, make_hashable

# User configs
from configs.paths import INTERIM_DIR, PROCESSED_DIR, SPLITS_DIR


### Load Inspected Data ###

In [2]:
otx = pd.read_parquet(INTERIM_DIR / "otx_inspected.parquet")
cve = pd.read_parquet(INTERIM_DIR / "cve_inspected.parquet")
domains = pd.read_parquet(INTERIM_DIR / "domains_inspected.parquet")
ips = pd.read_parquet(INTERIM_DIR / "ips_inspected.parquet")

### OTX Pipeline ###

In [3]:
def collapse_sub_technique(tech_id: str) -> str:
    """
    Maps: T1059.001, T1059.003 → T1059

    Keep base techniques intact.
    """

    tech_id = str(tech_id).strip().lower()

    if "." in tech_id:
        return tech_id.split(".")[0]
    
    return tech_id

def preprocess_attack_ids(attack_series: pd.Series) -> pd.Series:
    """
    Cleans, collapses, deduplicates ATT&CK IDs.

    Example:
        'T1059.001, T1027, T1059.003'
        ->
        ['T1027', 'T1059']
    """
    def process_row(val):

        if pd.isna(val) or not str(val).strip():
            return []

        # Split by comma/semicolon
        raw_ids = [
            x.strip()
            for x in str(val)
                .replace(";", ",")
                .split(",")
            if x.strip()
        ]

        # Collapse sub-techniques
        collapsed = [
            collapse_sub_technique(x)
            for x in raw_ids
        ]

        # Deduplicate + sort
        collapsed = sorted(list(set(collapsed)))

        return collapsed

    return attack_series.apply(process_row)

In [ ]:
# Configs
MIN_ATTACK_FREQ = 10        # Reduced from 20 to recover rare techniques
MAX_TAGS_LEN = 30           # Prevent metadata dominance

from configs.lookups import get_attack_keywords
ATTACK_KEYWORDS = get_attack_keywords()

TLP_MAP = {"white": 0, "green": 1, "amber": 2, "red": 3}

def preprocess_otx(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Text normalization
    text_cols = ["Title", "Description", "Tags", "Malware_Families", "Industries", "Countries"]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").map(normalize_text)

    # Time feature (most useful numeric signal)
    df["pulse_age_days"] = ((df["Modified"] - df["Created"]).dt.days.fillna(0).clip(lower=0))
    df["log_pulse_age"] = np.log1p(df["pulse_age_days"])

    # TLP encoding
    df["TLP_encoded"] = df["TLP"].astype(str).str.lower().map(TLP_MAP).fillna(0).astype(int)

    # Remove rows where Attack_List is ["unknown"] (189 noise rows)
    df = df[~df["Attack_IDs"].fillna("").astype(str).str.lower().str.contains("unknown", na=False)].reset_index(drop=True)

    # Malware family presence flags (top families as one-hot features)
    all_families = df["Malware_Families"].fillna("").str.split(",").explode().str.strip()
    top_families = all_families.value_counts().head(20).index.tolist()
    for fam in top_families:
        safe_col = f"malware_{fam}".replace(" ", "_").replace("-", "_").lower()
        df[safe_col] = df["Malware_Families"].fillna("").str.contains(
            re.escape(fam), case=False, na=False
        ).astype(int)

    # ATT&CK lexical keyword signals
    combined_lower = (
        df["Title"] + " " + df["Description"] + " " + df["Tags"]
    ).str.lower()
    for keyword, technique in ATTACK_KEYWORDS.items():
        safe_col = f"indicator_{technique}"
        df[safe_col] = combined_lower.str.contains(re.escape(keyword), na=False).astype(int)

    # ATT&CK multi-label
    df["Attack_List"] = preprocess_attack_ids(df["Attack_IDs"])

    # Add MITRE ATT&CK tactic mapping (technique → tactic as hierarchical signal)
    TECHNIQUE_TO_TACTIC = {
        "t1059": "ta0002",  # Command and Scripting Interpreter → Execution
        "t1204": "ta0005",  # User Execution → Defense Evasion
        "t1003": "ta0006",  # Credential Dumping → Credential Access
        "t1566": "ta0001",  # Phishing → Initial Access
        "t1547": "ta0003",  # Boot or Logon Autostart → Persistence
        "t1218": "ta0005",  # Signed Binary Proxy Execution → Defense Evasion
        "t1053": "ta0002",  # Scheduled Task → Execution
        "t1047": "ta0002",  # WMI → Execution
        "t1027": "ta0005",  # Obfuscated Files → Defense Evasion
        "t1036": "ta0005",  # Masquerading → Defense Evasion
        "t1071": "ta0011",  # Application Layer Protocol → Command and Control
        "t1140": "ta0005",  # Deobfuscate → Defense Evasion
        "t1055": "ta0005",  # Process Injection → Defense Evasion
        "t1082": "ta0007",  # System Information Discovery → Discovery
        "t1555": "ta0006",  # Credentials from Password Stores → Credential Access
        "t1005": "ta0007",  # Data from Local System → Collection
        "t1486": "ta0040",  # Data Encrypted for Impact → Impact
        "t1498": "ta0040",  # Network Denial of Service → Impact
        "t1499": "ta0040",  # Endpoint Denial of Service → Impact
        "t1529": "ta0040",  # System Shutdown/Reboot → Impact
        "t1041": "ta0011",  # Exfiltration Over C2 Channel → Exfiltration
        "t1114": "ta0007",  # Email Collection → Collection
    }
    df["tactic_labels"] = df["Attack_List"].apply(
        lambda x: list(set(TECHNIQUE_TO_TACTIC.get(t, "") for t in x if t in TECHNIQUE_TO_TACTIC))
    )

    # Filter rare labels
    counter = Counter(l for row in df["Attack_List"] for l in row)
    valid_labels = {k for k, v in counter.items() if v >= MIN_ATTACK_FREQ}
    df["Attack_List"] = df["Attack_List"].apply(lambda x: [i for i in x if i in valid_labels])

    # Remove empty rows
    df = df[df["Attack_List"].map(len) > 0].reset_index(drop=True)

    df["combined_text"] = (
        df["Title"] + " " +
        df["Description"] + " " +
        df["Tags"] + " " +
        df["Malware_Families"] + " " +
        df["Industries"] + " " +
        df["Countries"]
    ).str.replace(r"\s+", " ", regex=True).str.strip()

    # Convert lists to tuples for hashing
    df["Attack_List"] = df["Attack_List"].apply(tuple)

    # Remove duplicates
    df = df.drop_duplicates(subset=["combined_text", "Attack_List"])

    # Convert back to lists
    df["Attack_List"] = df["Attack_List"].apply(list)

    # Drop low-value / constant columns
    drop_cols = [c for c in ["Author", "Indicators_Count", "Subscribers",
                             "log_indicators", "log_subscribers", "Created_day"]
                if c in df.columns]
    df = df.drop(columns=drop_cols)

    return df

otx_p = preprocess_otx(otx)

In [5]:
# Remove redundant features
otx = otx.drop(columns=["Countries"])
otx_p

,Pulse_ID,Title,Description,Created,Modified,TLP,Tags,Malware_Families,Attack_IDs,Industries,...,indicator_t1204,indicator_t1003,indicator_t1566,indicator_t1059,indicator_t1547,indicator_t1218,indicator_t1053,indicator_t1047,Attack_List,combined_text
0,69f1e236e4e192f639298d53,multi-stage malware execution chain analysis,a sophisticated multi-stage malware execution ...,2026-04-29 10:49:26.327,2026-04-29 10:50:57.999,white,"payload extraction, c2 communication, defense ...",unknown,"T1036.005, T1082, T1071, T1140, T1036, T1055, ...",unknown,...,0,0,0,0,0,0,0,0,"[t1027, t1036, t1041, t1055, t1059, t1071, t10...",multi-stage malware execution chain analysis a...
1,69f1de85544538ce8b03332a,user interaction with a clickfix-style phishin...,a clickfix-style phishing campaign leveraged s...,2026-04-29 10:33:41.967,2026-04-29 10:44:36.742,white,"phishing, lumma stealer, powershell, informati...","hijackloader, lumma stealer - s1213, lummastealer","T1218.007, T1005, T1555, T1036, T1055, T1059, ...",unknown,...,0,0,1,1,0,0,0,0,"[t1005, t1027, t1036, t1041, t1055, t1059, t10...",user interaction with a clickfix-style phishin...
2,69f1d20216d6091f01f8a6eb,kyber ransomware is not just post-quantum name...,a detailed technical analysis confirms that ky...,2026-04-29 09:40:17.996,2026-04-29 10:14:38.724,white,"post-quantum cryptography, x25519, aes-ctr enc...",kyber,"T1489, T1135, T1082, T1112, T1070.001, T1222, ...","defense, technology",...,0,0,0,0,0,0,0,0,"[t1027, t1059, t1070, t1082, t1083, t1112, t11...",kyber ransomware is not just post-quantum name...
3,69f1d26f3c7a8e098eccb448,rebex-based telegram rat targeting vietnam,a sophisticated chm-based malware campaign has...,2026-04-29 09:42:07.871,2026-04-29 10:13:37.390,white,"multi-stage payload, telegram rat, chm infecti...",unknown,"T1053.005, T1036.005, T1204.002, T1497.001, T1...",unknown,...,0,0,0,0,0,0,0,0,"[t1027, t1036, t1053, t1059, t1071, t1082, t11...",rebex-based telegram rat targeting vietnam a s...
4,69f1d2d45ec26fc5e1ca72f4,kycshadow: an android banking malware exploiti...,an android malware campaign masquerading as a ...,2026-04-29 09:43:48.542,2026-04-29 10:12:57.758,white,"india targeting, android banking trojan, otp t...",kycshadow,Unknown,finance,...,0,0,0,0,0,0,0,0,[unknown],kycshadow: an android banking malware exploiti...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2347,66b22f6634824f0fa4c20d03,north korean hacking groups stealing construct...,"south koreas cybersecurity community, consisti...",2024-08-06 14:12:54.554,2024-09-05 14:02:22.746,white,"cyber attack, north korea, espionage, machiner...","trollagent, dorarat","T1113, T1074.001, T1204.002, T1119, T1005, T11...",unknown,...,0,0,0,0,0,0,0,0,"[t1005, t1027, t1036, t1041, t1071, t1074, t10...",north korean hacking groups stealing construct...
2348,66b1f5129138f4df25c4f4cf,blankbot: a new android banking trojan,a new android banking trojan called blankbot h...,2024-08-06 10:04:02.624,2024-09-05 10:02:03.154,white,"blankbot, android, trojan, banking, keylogging",blankbot,Unknown,unknown,...,0,0,0,0,0,0,0,0,[unknown],blankbot: a new android banking trojan a new a...
2349,66b0b7827198afc7407b35b2,stormbamboo compromises isp to abuse insecure ...,volexity detected and responded to multiple in...,2024-08-05 11:29:06.085,2024-09-04 11:01:46.096,white,"reloadext, dns poisoning, macma, osx.cdds, daz...","macma - s1016, osx.cdds, dazzlespy, pocostick,...","T1056.001, T1059.007, T1195, T1583.001, T1553....",unknown,...,0,0,0,0,0,0,0,0,"[t1056, t1059, t1071, t1078, t1195, t1199, t15...",stormbamboo compromises isp to abuse insecure ...
2350,66b090cdcc6b1efeb7afc7b9,checkmesh: hidden threats in your fw,this report examines an advanced cyber-attack ...,2024-08-05 08:43:57.589,2024-09-04 08:03:43.335,white,"meshagent, encrypted communication, advanced p...",meshagent,"T1543, T1547, T1082, T1053, T1005, T1190, T121...",unknown,...,0,0,0,0,0,0,0,0,"[t1005, t1021, t1027, t1053, t105

In [6]:
otx_p.describe()

,Created,Modified,Created_year,Created_month,pulse_age_days,log_pulse_age,TLP_encoded,malware_unknown,malware_cobalt_strike___s0154,malware_lumma_stealer,...,indicator_t1055,indicator_t1574,indicator_t1204,indicator_t1003,indicator_t1566,indicator_t1059,indicator_t1547,indicator_t1218,indicator_t1053,indicator_t1047
count,2351,2351,2351.000000,2351.000000,2352.000000,2352.000000,2352.000000,2352.000000,2352.000000,2352.000000,...,2352.000000,2352.000000,2352.000000,2352.000000,2352.000000,2352.000000,2352.000000,2352.000000,2352.000000,2352.000000
mean,2025-06-20 21:46:45.846985472,2025-07-16 14:24:43.075580416,2024.969800,6.492556,25.050595,1.711141,0.001276,0.276786,0.025935,0.021684,...,0.034864,0.025085,0.005527,0.008503,0.280187,0.092687,0.000850,0.001276,0.010629,0.000850
min,2015-01-19 18:20:52.140000,2024-09-04 08:03:43.335000,2015.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025-01-31 01:02:23.412000,2025-02-21 09:01:55.753999872,2025.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2025-06-26 17:27:51.777999872,2025-07-16 19:07:42.348999936,2025.000000,7.000000,2.000000,1.098612,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2025-11-26 23:06:06.332499968,2025-12-10 15:13:54.863500032,2025.000000,10.000000,29.000000,3.401197,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,2026-04-29 10:49:26.327000,2026-04-29 18:04:13.405000,2026.000000,12.000000,3802.000000,8.243546,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,NaN,NaN,0.754802,3.587545,142.362106,1.746823,0.035699,0.447505,0.158976,0.145679,...,0.183474,0.156417,0.074155,0.091841,0.449186,0.290055,0.029154,0.035699,0.102571,0.029154


### CVE Pipeline ###

In [7]:
def preprocess_cve(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Text cleanup
    text_cols = ["vulnerabilityName", "shortDescription", "requiredAction"]

    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").map(normalize_text)

    # Categorical cleanup
    cat_cols = ["vendorProject", "product", "cwes"]

    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.lower().str.strip()

    # Datetime
    df["dateAdded"] = pd.to_datetime(df["dateAdded"], errors="coerce")
    df["dueDate"] = pd.to_datetime(df["dueDate"], errors="coerce")

    df["days_to_due"] = (df["dueDate"] - df["dateAdded"]).dt.days
    df["days_to_due"] = df["days_to_due"].clip(-1, 3650)

    df["log_days_to_due"] = np.log1p(df["days_to_due"].clip(lower=0))

    # Recency feature
    ref_date = df["dateAdded"].max()
    df["days_since_added"] = (ref_date - df["dateAdded"]).dt.days.fillna(0).clip(lower=0)
    df["log_days_since_added"] = np.log1p(df["days_since_added"])

    # Text field (Lower Noise Design)
    df["risk_text"] = (df["vulnerabilityName"] + " " + df["shortDescription"]).str.strip()

    # requiredAction kept as separate structured feature (too repetitive for TF-IDF: 893/1585 identical)
    df["action_encoded"] = df["requiredAction"].astype(str).str.lower().map(
        lambda x: 1 if "apply" in x or "update" in x or "patch" in x else (
            2 if "mitigate" in x or "workaround" in x else (
                3 if "vendor" in x or "upstream" in x else 0
            )
        )
    ).fillna(0).astype(int)

    # Structured meta field
    df["meta_text"] = (df["vendorProject"] + " " + df["product"]).str.strip()

    # CWE risk weight scoring (OWASP-based severity per CWE)
    CWE_RISK_SCORE = {
        "cwe-22": 0.7, "cwe-77": 0.8, "cwe-79": 0.6, "cwe-89": 0.8, "cwe-94": 0.7,
        "cwe-119": 0.7, "cwe-120": 0.7, "cwe-125": 0.5, "cwe-200": 0.5, "cwe-287": 0.6,
        "cwe-306": 0.8, "cwe-352": 0.6, "cwe-416": 0.7, "cwe-434": 0.8, "cwe-502": 0.7,
        "cwe-787": 0.8, "cwe-862": 0.6, "cwe-918": 0.7, "unknown": 0.3
    }
    # Normalize CWE: strip "cwe-" prefix, lowercase, sort
    df["cwe_list"] = df["cwes"].astype(str).str.lower().str.strip().apply(
        lambda x: sorted(set(
            c.strip() for c in re.split(r"[,\s;]+", x) if c.strip() and c.strip() != "unknown"
        )) if x and x != "nan" else []
    )
    df["cwe_risk_max"] = df["cwe_list"].apply(
        lambda cwes: max([CWE_RISK_SCORE.get(c, 0.3) for c in (cwes or ["unknown"])])
    )
    df["cwe_count"] = df["cwe_list"].apply(len)

    return df

cve_p = preprocess_cve(cve)

In [8]:
cve_p

,cveID,vendorProject,product,vulnerabilityName,dateAdded,shortDescription,requiredAction,dueDate,knownRansomwareCampaignUse,cwes,dateAdded_year,dateAdded_month,dateAdded_day,cwe_list,days_to_due,log_days_to_due,days_since_added,log_days_since_added,risk_text,meta_text
0,CVE-2024-1708,connectwise,screenconnect,connectwise screenconnect path traversal vulne...,2026-04-28,connectwise screenconnect contains a path trav...,"apply mitigations per vendor instructions, fol...",2026-05-12,Unknown,cwe-22,2026,4,28,[CWE-22],14,2.708050,0,0.000000,connectwise screenconnect path traversal vulne...,connectwise screenconnect
1,CVE-2026-32202,microsoft,windows,microsoft windows protection mechanism failure...,2026-04-28,microsoft windows shell contains a protection ...,"apply mitigations per vendor instructions, fol...",2026-05-12,Unknown,cwe-693,2026,4,28,[CWE-693],14,2.708050,0,0.000000,microsoft windows protection mechanism failure...,microsoft windows
2,CVE-2025-29635,d-link,dir-823x,d-link dir-823x command injection vulnerability,2026-04-24,d-link dir-823x contains a command injection v...,"apply mitigations per vendor instructions, fol...",2026-05-08,Unknown,cwe-77,2026,4,24,[CWE-77],14,2.708050,4,1.609438,d-link dir-823x command injection vulnerabilit...,d-link dir-823x
3,CVE-2024-7399,samsung,magicinfo 9 server,samsung magicinfo 9 server path traversal vuln...,2026-04-24,samsung magicinfo 9 server contains a path tra...,"apply mitigations per vendor instructions, fol...",2026-05-08,Unknown,"cwe-22, cwe-434",2026,4,24,"[CWE-22, CWE-434]",14,2.708050,4,1.609438,samsung magicinfo 9 server path traversal vuln...,samsung magicinfo 9 server
4,CVE-2024-57728,simplehelp,simplehelp,simplehelp path traversal vulnerability,2026-04-24,simplehelp contains a path traversal vulnerabi...,"apply mitigations per vendor instructions, fol...",2026-05-08,Unknown,cwe-22,2026,4,24,[CWE-22],14,2.708050,4,1.609438,simplehelp path traversal vulnerability simple...,simplehelp simplehelp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1580,CVE-2021-27561,yealink,device management,yealink device management server-side request ...,2021-11-03,yealink device management contains a server-si...,apply updates per vendor instructions.,2021-11-17,Unknown,cwe-78,2021,11,3,[CWE-78],14,2.708050,1637,7.401231,yealink device management server-side request ...,yealink device management
1581,CVE-2021-40539,zoho,manageengine,zoho manageengine adselfservice plus authentic...,2021-11-03,zoho manageengine adselfservice plus contains ...,apply updates per vendor instructions.,2021-11-17,Known,cwe-55,2021,11,3,[CWE-55],14,2.708050,1637,7.401231,zoho manageengine adselfservice plus authentic...,zoho manageengine
1582,CVE-2020-10189,zoho,manageengine,zoho manageengine desktop central file upload ...,2021-11-03,zoho manageengine desktop central contains a f...,apply updates per vendor instructions.,2022-05-03,Unknown,cwe-502,2021,11,3,[CWE-502],181,5.204007,1637,7.401231,zoho manageengine desktop central file upload ...,zoho manageengine
1583,CVE-2019-8394,zoho,manageengine,zoho manageengine servicedesk plus sdp file up...,2021-11-03,zoho manageengine servicedesk plus sdp contain...,apply updates per vendor instructions.,2022-05-03,Unknown,cwe-434,2021,11,3,[CWE-434],181,5.204007,1637,7.401231,zoho manageengine servicedesk plus sdp file up...,zoho manageengine


In [9]:
# Get unique CVE values from df["cve"]
unique_cves = cve_p["cwes"].unique()

# Convert to list if needed
unique_cve_list = unique_cves.tolist()

print(unique_cve_list)

['cwe-22', 'cwe-693', 'cwe-77', 'cwe-22, cwe-434', 'cwe-862', 'cwe-306', 'cwe-1220', 'cwe-648', 'cwe-200', 'cwe-287', 'cwe-79', 'cwe-257', 'cwe-23', 'cwe-20, cwe-94', 'cwe-94', 'cwe-20', 'cwe-426', 'cwe-59', 'cwe-502', 'cwe-125', 'cwe-416', 'cwe-89', 'cwe-1321', 'cwe-284', 'cwe-494', 'cwe-121', 'cwe-506', 'cwe-94, cwe-95, cwe-306', 'cwe-667', 'cwe-120', 'cwe-119', 'cwe-209', 'cwe-787', 'cwe-913', 'cwe-918', 'cwe-288', 'cwe-522', 'cwe-190', 'cwe-25, cwe-282', 'cwe-78', 'cwe-798', 'cwe-434', 'unknown', 'cwe-476', 'cwe-269', 'cwe-843', 'cwe-807', 'cwe-88', 'cwe-98', 'cwe-200, cwe-284', 'cwe-130', 'cwe-862, cwe-250', 'cwe-347', 'cwe-611', 'cwe-362', 'cwe-552', 'cwe-267', 'cwe-95', 'cwe-940', 'cwe-324', 'cwe-822', 'cwe-306, cwe-77', 'cwe-829', 'cwe-502, cwe-77', 'cwe-367', 'cwe-290', 'cwe-863', 'cwe-89, cwe-288', 'cwe-59, cwe-436', 'cwe-35', 'cwe-399', 'cwe-352', 'cwe-74', 'cwe-420', 'cwe-158', 'cwe-918, cwe-807', 'cwe-77, cwe-88', 'cwe-528', 'cwe-1188', 'cwe-282', 'cwe-73', 'cwe-125, cwe-7

In [10]:
cve_p.describe()

,dateAdded,dueDate,dateAdded_year,dateAdded_month,dateAdded_day,days_to_due,log_days_to_due,days_since_added,log_days_since_added
count,1585,1585,1585.000000,1585.000000,1585.000000,1585.000000,1585.000000,1585.000000,1585.000000
mean,2023-05-13 00:20:53.753943296,2023-06-27 08:42:23.848580352,2022.875079,6.459306,13.076341,45.348265,3.322813,1080.985489,6.687181
min,2021-11-03 00:00:00,2021-11-17 00:00:00,2021.000000,1.000000,1.000000,1.000000,0.693147,0.000000,0.000000
25%,2022-03-03 00:00:00,2022-05-03 00:00:00,2022.000000,3.000000,3.000000,21.000000,3.091042,567.000000,6.342121
50%,2022-08-11 00:00:00,2022-09-07 00:00:00,2022.000000,6.000000,11.000000,21.000000,3.091042,1356.000000,7.213032
75%,2024-10-08 00:00:00,2024-10-29 00:00:00,2024.000000,11.000000,22.000000,21.000000,3.091042,1517.000000,7.325149
max,2026-04-28 00:00:00,2026-05-12 00:00:00,2026.000000,12.000000,31.000000,184.000000,5.220356,1637.000000,7.401231
std,NaN,NaN,1.558875,3.651451,9.174300,59.880541,0.885363,546.839487,1.044337


### Domain Pipeline ###

In [11]:
def _domain_entropy(domain: str) -> float:
    if not isinstance(domain, str) or len(domain) == 0:
        return 0.0
    prob = [float(domain.count(c)) / len(domain) for c in set(domain)]
    return -sum(p * math.log(p, 2) for p in prob)

from configs.lookups import get_suspicious_keywords, get_high_risk_tlds, get_brand_keywords
# Lookups are called INSIDE preprocess_domains() — not at module level
# See the function body for the internal API calls

def preprocess_domains(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Internal API: configs/lookups loaded inside the function
    from configs.lookups import get_suspicious_keywords, get_high_risk_tlds, get_brand_keywords
    SUSPICIOUS_KEYWORDS = get_suspicious_keywords()
    HIGH_RISK_TLDS = get_high_risk_tlds()
    BRAND_KEYWORDS = get_brand_keywords()

    # Numeric
    numeric_cols = [
        "Domain_Length", "Reputation", "Malicious_Votes",
        "Suspicious_Votes", "Harmless_Votes", "Undetected_Votes",
        "Total_Engines"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # Store raw domain string for char-level TF-IDF
    df["domain_string"] = df["Domain"].astype(str).str.lower().str.strip()

    # Date fix
    df["Creation_Date"] = pd.to_datetime(df["Creation_Date"], errors="coerce")
    df["Last_Analysis_Date"] = pd.to_datetime(df["Last_Analysis_Date"], errors="coerce")

    df["domain_age_days"] = (
        df["Last_Analysis_Date"] - df["Creation_Date"]
    ).dt.days.fillna(0).clip(lower=0)

    df["log_domain_age"] = np.log1p(df["domain_age_days"])
    df["is_new_domain"] = (df["domain_age_days"] < 30).astype(int)

    # Engineered binary flags
    df["Has_Numbers"] = df["Domain"].str.contains(r"\d", na=False).astype(int)
    df["Has_Hyphen"] = df["Domain"].str.contains(r"-", na=False).astype(int)

    # Lexical domain intelligence features
    df["entropy"] = df["Domain"].astype(str).apply(_domain_entropy)
    df["digit_ratio"] = df["Domain"].astype(str).apply(
        lambda d: sum(c.isdigit() for c in str(d)) / max(len(str(d)), 1)
    )
    df["vowel_ratio"] = df["Domain"].astype(str).apply(
        lambda d: sum(c in "aeiou" for c in str(d).lower()) / max(len(str(d)), 1)
    )
    df["special_ratio"] = df["Domain"].astype(str).apply(
        lambda d: sum(not c.isalnum() for c in str(d)) / max(len(str(d)), 1)
    )
    df["subdomain_count"] = df["Domain"].astype(str).apply(
        lambda d: max(len(str(d).split(".")) - 1, 0)
    )
    df["token_count"] = df["Domain"].astype(str).apply(
        lambda d: len(re.split(r"[.-]", str(d)))
    )
    df["max_token_length"] = df["Domain"].astype(str).apply(
        lambda d: max((len(t) for t in re.split(r"[.-]", str(d))), default=0)
    )
    df["consecutive_consonants"] = df["Domain"].astype(str).apply(
        lambda d: max((len(m) for m in re.findall(r"[bcdfghjklmnpqrstvwxyz]{2,}", str(d).lower())), default=0)
    )
    df["consecutive_digits"] = df["Domain"].astype(str).apply(
        lambda d: max((len(m) for m in re.findall(r"\d{2,}", str(d))), default=0)
    )

    # Suspicious keyword flags
    domain_lower = df["Domain"].astype(str).str.lower()
    df["suspicious_keyword_count"] = domain_lower.apply(
        lambda d: sum(1 for kw in SUSPICIOUS_KEYWORDS if kw in d)
    )
    df["contains_brand_keyword"] = domain_lower.apply(
        lambda d: int(any(bk in d for bk in BRAND_KEYWORDS))
    )
    df["contains_login_keyword"] = domain_lower.str.contains(
        r"login|signin|account|credential", na=False
    ).astype(int)
    df["contains_crypto_keyword"] = domain_lower.str.contains(
        r"crypto|wallet|bitcoin|blockchain", na=False
    ).astype(int)
    df["contains_bank_keyword"] = domain_lower.str.contains(
        r"bank|paypal|transfer|payment", na=False
    ).astype(int)
    df["is_randomized_domain"] = (
        (df["entropy"] > 3.5) & (df["digit_ratio"] > 0.3)
    ).astype(int)

    # Ratios
    df["malicious_ratio"] = safe_ratio(df["Malicious_Votes"], df["Total_Engines"])
    df["suspicious_ratio"] = safe_ratio(df["Suspicious_Votes"], df["Total_Engines"])

    # Log counts
    df["log_malicious"] = np.log1p(df["Malicious_Votes"])
    df["log_suspicious"] = np.log1p(df["Suspicious_Votes"])

    # Categorical cleaning
    df["TLD"] = df["TLD"].str.lower().fillna("unknown")
    df["tld_risk_score"] = df["TLD"].apply(
        lambda t: 2 if str(t) in {t.replace(".", "") for t in HIGH_RISK_TLDS}
        else 1 if str(t) in {"com", "net", "org"}
        else 0
    )

    # WHOIS completeness
    df["has_creation_date"] = df["Creation_Date"].notna().astype(int)
    df["has_registrar"] = (df["Registrar"].astype(str).str.lower() != "unknown").astype(int)
    df["has_nameservers"] = df.get("Name_Servers", pd.Series([np.nan] * len(df))).notna().astype(int)
    whois_fields = ["Creation_Date", "Last_Update_Date", "Registrar", "Name_Servers", "WHOIS_Summary"]
    df["whois_field_count"] = df[[c for c in whois_fields if c in df.columns]].notna().sum(axis=1)

    # Remove noisy / weak columns from modeling scope (keep in df for reference)
    # Popularity_Rank, Data_Source, Registrar are high-sparsity / leakage risk

    return df

domains_p = preprocess_domains(domains)

In [12]:
domains_p

,Domain,TLD,Domain_Length,Has_Numbers,Has_Hyphen,Registrar,Creation_Date,Last_Update_Date,Reputation,Malicious_Votes,...,is_randomized_domain,malicious_ratio,suspicious_ratio,log_malicious,log_suspicious,tld_risk_score,has_creation_date,has_registrar,has_nameservers,whois_field_count
0,urlhaus.abuse.ch,ch,16,0,0,Unknown,NaT,NaT,2,0,...,0,0.000000,0.000000,0.000000,0.000000,0,0,0,0,2
1,bazaar.abuse.ch,ch,15,0,0,Unknown,NaT,NaT,6,1,...,0,0.010989,0.000000,0.693147,0.000000,0,0,0,0,2
2,feodotracker.abuse.ch,ch,21,0,0,Unknown,NaT,NaT,1,0,...,0,0.000000,0.000000,0.000000,0.000000,0,0,0,0,2
3,sslbl.abuse.ch,ch,14,0,0,Unknown,NaT,NaT,0,0,...,0,0.000000,0.000000,0.000000,0.000000,0,0,0,0,2
4,threatfox.abuse.ch,ch,18,0,0,Unknown,NaT,NaT,0,0,...,0,0.000000,0.000000,0.000000,0.000000,0,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,adobe-update.com,com,16,0,1,"GMO Internet Group, Inc. d/b/a Onamae.com",2024-10-05 09:08:21,2025-10-06 07:36:49,0,6,...,0,0.065934,0.010989,1.945910,0.693147,1,1,1,0,4
158,windows-update.net,net,18,0,1,"NameSilo, LLC",2025-02-18 06:35:04,2026-02-10 09:31:00,0,0,...,0,0.000000,0.000000,0.000000,0.000000,1,1,1,0,4
159,antivirus-update.org,org,20,0,1,Unknown,NaT,NaT,0,0,...,0,0.000000,0.000000,0.000000,0.000000,1,0,0,0,2
160,driver-update.net,net,17,0,1,"GoDaddy.com, LLC",2008-05-11 17:39:20,2023-04-24 21:44:23,0,0,...,0,0.000000,0.000000,0.000000,0.000000,1,1,1,0,4


In [19]:
domains_p.describe()

,Domain_Length,Has_Numbers,Has_Hyphen,Creation_Date,Last_Update_Date,Reputation,Malicious_Votes,Suspicious_Votes,Harmless_Votes,Undetected_Votes,...,is_randomized_domain,malicious_ratio,suspicious_ratio,log_malicious,log_suspicious,tld_risk_score,has_creation_date,has_registrar,has_nameservers,whois_field_count
count,162.000000,162.00000,162.000000,120,128,162.000000,162.000000,162.000000,162.000000,162.000000,...,162.0,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.0,162.000000
mean,15.820988,0.04321,0.537037,2017-10-03 15:53:55.516666624,2024-03-31 00:56:53.250000128,0.654321,1.308642,0.228395,43.209877,46.253086,...,0.0,0.014381,0.002510,0.404551,0.150481,0.765432,0.740741,0.555556,0.0,3.530864
min,6.000000,0.00000,0.000000,2002-07-17 22:54:40,2014-12-04 15:13:36,-47.000000,0.000000,0.000000,0.000000,29.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,2.000000
25%,12.000000,0.00000,0.000000,2014-06-13 02:26:41,2023-09-20 21:20:21.750000128,0.000000,0.000000,0.000000,46.000000,33.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,3.000000
50%,16.000000,0.00000,1.000000,2017-10-10 20:55:14,2025-07-02 14:39:55.500000,0.000000,0.000000,0.000000,55.000000,34.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.0,4.000000
75%,19.000000,0.00000,1.000000,2023-01-31 18:00:00,2025-11-29 18:00:00,0.000000,1.000000,0.000000,57.000000,41.750000,...,0.0,0.010989,0.000000,0.693147,0.000000,1.000000,1.000000,1.000000,0.0,4.000000
max,28.000000,1.00000,1.000000,2026-04-02 00:00:00,2026-04-05 05:23:09,140.000000,17.000000,3.000000,61.000000,91.000000,...,0.0,0.186813,0.032967,2.890372,1.386294,2.000000,1.000000,1.000000,0.0,4.000000
std,4.748199,0.20396,0.500173,NaN,NaN,13.143886,3.340480,0.489323,23.030747,23.698759,...,0.0,0.036709,0.005377,0.750302,0.306056,0.453327,0.439587,0.498445,0.0,0.812731


### IP Pipeline ###

In [13]:
from configs.lookups import get_high_risk_countries, get_known_malicious_asns
HIGH_RISK_COUNTRIES = get_high_risk_countries()

KNOWN_MALICIOUS_ASNS = get_known_malicious_asns()

def preprocess_ips(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Text (NO Threat_Label / Threat_Category — they leak the target)
    text_cols = [
        "Country", "Continent", "Owner", "Network",
        "Regional_Registry", "WHOIS_Summary"
    ]

    # Drop constant / zero-variance columns (Total_Reports=91 for all, Times_Submitted=0 for all)
    constant_cols = [c for c in ["Total_Reports", "Times_Submitted"] if c in df.columns]
    if constant_cols:
        df = df.drop(columns=constant_cols)

    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").map(normalize_text)

    # Numeric (excluding constant / near-zero variance columns)
    numeric_cols = [
        "Malicious_Votes", "Suspicious_Votes",
        "Harmless_Votes", "Undetected_Votes",
        "Reputation_Score"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # ASN as categorical
    df["ASN"] = df["ASN"].astype(str).fillna("unknown")

    # Merge rare ASNs (freq < 3) into "other" category to reduce overfitting (31 unique → ~10)
    asn_counts = df["ASN"].value_counts()
    rare_asns = asn_counts[asn_counts < 3].index
    df["ASN"] = df["ASN"].apply(lambda x: "other" if x in rare_asns else x)

    df["asn_risk_flag"] = df["ASN"].isin(KNOWN_MALICIOUS_ASNS).astype(int)

    # Ratios (using raw votes only, Total_Reports is constant)
    total = df["Malicious_Votes"] + df["Suspicious_Votes"] + df["Harmless_Votes"] + df["Undetected_Votes"]
    df["malicious_ratio"] = safe_ratio(df["Malicious_Votes"], total)
    df["suspicious_ratio"] = safe_ratio(df["Suspicious_Votes"], total)

    # Log features
    df["log_malicious"] = np.log1p(df["Malicious_Votes"])
    df["log_suspicious"] = np.log1p(df["Suspicious_Votes"])
    df["log_harmless"] = np.log1p(df["Harmless_Votes"])
    df["log_undetected"] = np.log1p(df["Undetected_Votes"])

    # Safer reputation scaling
    min_val = df["Reputation_Score"].min()
    max_val = df["Reputation_Score"].max()

    df["reputation_score_scaled"] = (
        (df["Reputation_Score"] - min_val) /
        (max_val - min_val + 1e-9)
    )

    # TOR flag robustness
    df["tor_flag"] = df["TOR_Node"].astype(str).str.lower().isin(
        ["1", "true", "yes"]
    ).astype(int)

    # IP structure feature
    df["ip_first_octet"] = df["IP"].astype(str).str.split(".").str[0]

    # Geolocation risk
    df["high_risk_country"] = df["Country"].astype(str).str.lower().isin(HIGH_RISK_COUNTRIES).astype(int)
    df["unknown_continent"] = (df["Continent"].astype(str).str.lower() == "unknown").astype(int)

    # Reputation flags
    df["negative_reputation"] = (df["Reputation_Score"] < 0).astype(int)
    df["zero_votes"] = ((df["Malicious_Votes"] + df["Suspicious_Votes"] +
                          df["Harmless_Votes"] + df["Undetected_Votes"]) == 0).astype(int)

    return df

ips_p = preprocess_ips(ips)

In [14]:
ips_p

,IP,Country,Continent,ASN,Owner,Network,Malicious_Votes,Suspicious_Votes,Harmless_Votes,Undetected_Votes,...,log_suspicious,log_harmless,log_undetected,reputation_score_scaled,tor_flag,ip_first_octet,high_risk_country,unknown_continent,negative_reputation,zero_votes
0,176.10.99.200,ch,eu,51395.0,datasource ag,176.10.96.0/19,3,0,55,33,...,0.000000,4.025352,3.526361,0.686047,0,176,0,0,1,0
1,176.10.99.201,ch,eu,51395.0,datasource ag,176.10.96.0/19,0,1,53,37,...,0.693147,3.988984,3.637586,0.988372,0,176,0,0,0,0
2,77.247.181.162,nl,eu,43350.0,nforce entertainment b.v.,77.247.176.0/21,4,1,53,33,...,0.693147,3.988984,3.526361,0.767442,0,77,0,0,1,0
3,77.247.181.163,nl,eu,43350.0,nforce entertainment b.v.,77.247.176.0/21,2,2,55,32,...,1.098612,4.025352,3.496508,0.802326,0,77,0,0,1,0
4,199.87.154.255,ca,unknown,18451.0,les.net,199.87.152.0/21,2,0,54,35,...,0.000000,4.007333,3.583519,0.976744,0,199,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,45.142.122.103,vg,unknown,205090.0,first server limited,45.142.122.0/24,0,0,0,91,...,0.000000,0.000000,4.521789,0.988372,0,45,0,1,0,0
196,194.40.243.100,ua,eu,48693.0,rices privately owned enterprise,194.40.243.0/24,1,0,53,37,...,0.000000,3.988984,3.637586,0.988372,0,194,0,0,0,0
197,194.40.243.101,ua,eu,48693.0,rices privately owned enterprise,194.40.243.0/24,0,0,54,37,...,0.000000,4.007333,3.637586,0.988372,0,194,0,0,0,0
198,194.40.243.102,ua,eu,48693.0,rices privately owned enterprise,194.40.243.0/24,0,0,0,91,...,0.000000,0.000000,4.521789,0.988372,0,194,0,0,0,0


In [21]:
ips_p.describe()

,Malicious_Votes,Suspicious_Votes,Harmless_Votes,Undetected_Votes,Total_Reports,Reputation_Score,Times_Submitted,Last_Analysis_Date,Last_Analysis_Date_year,Last_Analysis_Date_month,...,log_malicious,log_suspicious,log_harmless,log_undetected,reputation_score_scaled,tor_flag,high_risk_country,unknown_continent,negative_reputation,zero_votes
count,200.000000,200.000000,200.000000,200.000000,200.0,200.000000,200.0,200,200.000000,200.000000,...,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.0
mean,1.500000,0.295000,24.285000,64.920000,91.0,-2.020000,0.0,2025-01-07 02:25:32.769999872,2024.650000,4.995000,...,0.418897,0.165449,1.853090,4.082866,0.964884,0.055000,0.080000,0.240000,0.135000,0.0
min,0.000000,0.000000,0.000000,28.000000,91.0,-85.000000,0.0,2018-12-26 14:02:21,2018.000000,1.000000,...,0.000000,0.000000,0.000000,3.367296,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
25%,0.000000,0.000000,0.000000,35.000000,91.0,0.000000,0.0,2024-04-17 08:28:53,2024.000000,4.000000,...,0.000000,0.000000,0.000000,3.583519,0.988372,0.000000,0.000000,0.000000,0.000000,0.0
50%,0.000000,0.000000,0.000000,91.000000,91.0,0.000000,0.0,2026-02-11 22:13:19.500000,2026.000000,4.500000,...,0.000000,0.000000,0.000000,4.521789,0.988372,0.000000,0.000000,0.000000,0.000000,0.0
75%,1.000000,0.000000,54.000000,91.000000,91.0,0.000000,0.0,2026-04-28 04:43:15.750000128,2026.000000,5.000000,...,0.693147,0.000000,4.007333,4.521789,0.988372,0.000000,0.000000,0.000000,0.000000,0.0
max,21.000000,4.000000,58.000000,91.000000,91.0,1.000000,0.0,2026-05-01 10:44:10,2026.000000,12.000000,...,3.091042,1.609438,4.077537,4.521789,1.000000,1.000000,1.000000,1.000000,1.000000,0.0
std,3.692992,0.742111,26.276148,28.099493,0.0,8.546038,0.0,NaN,1.922493,2.352156,...,0.814499,0.377332,1.980896,0.475321,0.099373,0.228552,0.271974,0.428155,0.342581,0.0


In [15]:
numerical_columns = ips_p.select_dtypes(
    include=["number"]
).columns.tolist()

print(numerical_columns)

['Malicious_Votes', 'Suspicious_Votes', 'Harmless_Votes', 'Undetected_Votes', 'Total_Reports', 'Reputation_Score', 'Times_Submitted', 'Last_Analysis_Date_year', 'Last_Analysis_Date_month', 'Last_Analysis_Date_day', 'asn_risk_flag', 'malicious_ratio', 'suspicious_ratio', 'log_malicious', 'log_suspicious', 'log_harmless', 'log_undetected', 'reputation_score_scaled', 'tor_flag', 'high_risk_country', 'unknown_continent', 'negative_reputation', 'zero_votes']


### Final Cleanup ###

In [16]:
def finalize_df(df):

    df = df.copy()

    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    
    for col in obj_cols:
        df[col] = df[col].map(make_hashable)

    df = df.drop_duplicates()

    for col in obj_cols:
        df[col] = df[col].fillna("")

    return df

otx_p = finalize_df(otx_p)
cve_p = finalize_df(cve_p)
domains_p = finalize_df(domains_p)
ips_p = finalize_df(ips_p)

### Savestates ###

In [17]:
otx_p.to_parquet(PROCESSED_DIR / "otx_processed.parquet", index=False)
cve_p.to_parquet(PROCESSED_DIR / "cve_processed.parquet", index=False)
domains_p.to_parquet(PROCESSED_DIR / "domains_processed.parquet", index=False)
ips_p.to_parquet(PROCESSED_DIR / "ips_processed.parquet", index=False)

### Split Dataframes ###

In [18]:
datasets = {
    "otx": otx_p,
    "cve": cve_p,
    "domains": domains_p,
    "ips": ips_p
}

for name, df in datasets.items():
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

    train_df.to_parquet(SPLITS_DIR / f"{name}_train.parquet", index=False)
    test_df.to_parquet(SPLITS_DIR / f"{name}_test.parquet", index=False)